# Stage 6 · Multi-Turn, Tools & Environment RL — SOLUTION
### Topics: Trajectory MDPs · Tool Actions · Budgets · POMDP-style Observations · Rollouts · Credit Assignment


In [ ]:
import json
import re
from dataclasses import dataclass, field
from typing import Any, Callable, Dict, List, Optional, Tuple

import numpy as np


---
## 1 · Episodes, transitions & Monte Carlo returns

Single-completion RLHF scores one string. **Environment RL** scores a **trajectory**:
$$\tau = (o_0, a_0, r_0, o_1, a_1, r_1, \dots)$$

The discounted return from timestep $t$ is:
$$G_t = \sum_{k=0}^{T-t} \gamma^k r_{t+k}$$

We compute $\{G_t\}$ with one backward pass (same trick as Stage 1). The **total return** from the start is $G_0$.


In [ ]:
@dataclass
class Transition:
    observation: str
    action:      str
    reward:      float
    done:        bool
    info:        Dict[str, Any] = field(default_factory=dict)


def compute_mc_returns(rewards: List[float], gamma: float) -> List[float]:
    """Monte Carlo returns G_t = r_t + γ G_{t+1}, computed backwards."""
    T = len(rewards)
    G = [0.0] * T
    acc = 0.0
    for t in reversed(range(T)):
        acc = rewards[t] + gamma * acc
        G[t] = acc
    return G


def total_return_from_rewards(rewards: List[float], gamma: float) -> float:
    """Scalar G_0 for an episode reward list."""
    if not rewards:
        return 0.0
    return compute_mc_returns(rewards, gamma)[0]


# ── Sanity checks ─────────────────────────────────────────────────────────
assert compute_mc_returns([1.0, 2.0, 3.0], 0.0) == [1.0, 2.0, 3.0]
assert abs(compute_mc_returns([1.0, 2.0, 3.0], 1.0)[0] - 6.0) < 1e-6
g = compute_mc_returns([0.0, 0.0, 1.0], 1.0)
assert abs(g[0] - 1.0) < 1e-6 and abs(g[1] - 1.0) < 1e-6 and abs(g[2] - 1.0) < 1e-6
print("compute_mc_returns ✓  total_return_from_rewards ✓")


---
## 2 · A minimal tool protocol (parse → execute in the env)

We standardise assistant **actions** as plain text with two shapes:
- **Tool call:** a single line starting with `TOOL ` followed by a JSON object.
- **Final answer:** `ANSWER <float>` (case-insensitive `ANSWER`).

In real systems, tool calls are often structured tokens; text JSON keeps the notebook dependency-free.


In [ ]:
def parse_tool_json(action: str) -> Optional[Dict[str, Any]]:
    """If action is `TOOL {...}` return the dict, else None."""
    a = action.strip()
    if not a.upper().startswith("TOOL"):
        return None
    rest = a[4:].strip()
    if not rest:
        return None
    try:
        return json.loads(rest)
    except json.JSONDecodeError:
        return None


def parse_answer_float(action: str) -> Optional[float]:
    """Parse `ANSWER 3.14` (flexible spacing / case)."""
    m = re.search(r"ANSWER\s*([+-]?\d*\.?\d+(?:[eE][+-]?\d+)?)", action.strip(), re.I)
    if not m:
        return None
    try:
        return float(m.group(1))
    except ValueError:
        return None


# ── Sanity checks ─────────────────────────────────────────────────────────
assert parse_tool_json('TOOL {"name":"lookup","key":"banana"}') == {"name": "lookup", "key": "banana"}
assert parse_tool_json("say hello") is None
assert abs(parse_answer_float("  ANSWER   0.5  ") - 0.5) < 1e-9
print("parse_tool_json ✓  parse_answer_float ✓")


---
## 3 · `ToyPricingEnv` — partial observability, budgets, shaped rewards

**Hidden state:** a price table and a target product key.  
**Observation:** instructions + question + (only if `pomdp=False`) an oracle list of valid keys.  
**Rewards:** small bonus for a successful `lookup`, +1 for the correct `ANSWER`, tiny penalties for malformed actions.

This is deliberately small so you can **unit-test credit assignment** without a language model.


In [ ]:
class ToyPricingEnv:
    """Single-question pricing task with a lookup tool."""

    def __init__(
        self,
        pomdp:          bool = True,
        max_turns:      int  = 8,
        max_tool_calls: int  = 3,
        tool_reward:    float = 0.1,
    ):
        self.pomdp = pomdp
        self.max_turns = max_turns
        self.max_tool_calls = max_tool_calls
        self.tool_reward = tool_reward
        self._prices = {"apple": 1.2, "banana": 0.5, "cherry": 2.0}
        self._target_key = "banana"
        self._answer = self._prices[self._target_key]
        self._turn = 0
        self._tool_calls = 0
        self._last_obs = ""

    def reset(self, question: Optional[str] = None) -> str:
        self._turn = 0
        self._tool_calls = 0
        q = question or "What is the price of banana (USD)?"
        lines = [
            "You are a pricing assistant.",
            "Call lookup with: TOOL {\"name\":\"lookup\",\"key\":\"<product>\"}",
            "Finish with: ANSWER <number>",
            f"Question: {q}",
        ]
        if not self.pomdp:
            lines.append("ORACLE: valid keys are " + ", ".join(sorted(self._prices.keys())) + ".")
        self._last_obs = "\n".join(lines)
        return self._last_obs

    def step(self, action: str) -> Tuple[str, float, bool, Dict[str, Any]]:
        self._turn += 1
        info: Dict[str, Any] = {"turn": self._turn, "tool_calls": self._tool_calls}

        def finish(obs: str, reward: float, done: bool, **extra: Any) -> Tuple[str, float, bool, Dict[str, Any]]:
            info.update(extra)
            self._last_obs = obs
            return obs, reward, done, info

        if self._turn > self.max_turns:
            return finish("Truncated: max turns.", 0.0, True, truncated=True)

        tool = parse_tool_json(action)
        if tool is not None:
            if self._tool_calls >= self.max_tool_calls:
                return finish("Tool budget exhausted.", -0.05, True, truncated=True, reason="tool_budget")
            self._tool_calls += 1
            if tool.get("name") != "lookup" or "key" not in tool:
                return finish("Unknown tool.", -0.05, False)
            key = str(tool["key"])
            if key in self._prices:
                val = self._prices[key]
                return finish(f"Tool result: {key} = {val}", self.tool_reward, False, tool_ok=True)
            return finish(f"Tool result: unknown key '{key}'", 0.0, False, tool_ok=False)

        ans = parse_answer_float(action)
        if ans is not None:
            if abs(ans - self._answer) < 1e-5:
                return finish("Correct.", 1.0, True, terminated=True, correct=True)
            return finish("Wrong answer.", -0.05, False, correct=False)

        return finish("Unrecognised action (use TOOL JSON or ANSWER).", -0.01, False)


# ── Sanity checks ─────────────────────────────────────────────────────────
env = ToyPricingEnv(pomdp=True)
obs0 = env.reset()
assert "banana" in obs0.lower() and "ORACLE" not in obs0
obs1, r1, d1, i1 = env.step('TOOL {"name":"lookup","key":"banana"}')
assert not d1 and abs(r1 - 0.1) < 1e-9 and "0.5" in obs1
obs2, r2, d2, i2 = env.step("ANSWER 0.5")
assert d2 and abs(r2 - 1.0) < 1e-9 and i2.get("terminated")

env_o = ToyPricingEnv(pomdp=False)
obs_o = env_o.reset()
assert "ORACLE" in obs_o and "apple" in obs_o

env_b = ToyPricingEnv(pomdp=True, max_tool_calls=1)
env_b.reset()
env_b.step('TOOL {"name":"lookup","key":"banana"}')
obs_b, r_b, d_b, i_b = env_b.step('TOOL {"name":"lookup","key":"apple"}')
assert d_b and i_b.get("truncated") and i_b.get("reason") == "tool_budget"
print("ToyPricingEnv ✓  (pomdp / oracle / tool budget)")


---
## 4 · Rollouts: collect trajectories from any policy

A **policy** maps the observation history (list of user/assistant strings) to the next action string.  
This is the same outer loop used in on-policy RL for tool-using LMs — only the policy is swapped out.


In [ ]:
PolicyFn = Callable[[List[str]], str]


def rollout_episode(
    env:     ToyPricingEnv,
    policy:  PolicyFn,
    max_steps: int = 16,
) -> List[Transition]:
    """Start from env.reset(); repeatedly append obs to history, policy → action, env.step until done."""
    history: List[str] = []
    obs = env.reset()
    history.append(obs)
    transitions: List[Transition] = []
    for _ in range(max_steps):
        action = policy(history)
        next_obs, reward, done, info = env.step(action)
        transitions.append(Transition(observation=obs, action=action, reward=reward, done=done, info=info))
        if done:
            break
        obs = next_obs
        history.append(obs)
    return transitions


def scripted_policy(actions: List[str]) -> PolicyFn:
    """Return a policy that emits actions in order (raises if exhausted)."""
    i = {"k": 0}

    def _pol(_hist: List[str]) -> str:
        k = i["k"]
        if k >= len(actions):
            raise RuntimeError("scripted_policy: no more actions")
        i["k"] = k + 1
        return actions[k]

    return _pol


# ── Sanity checks ─────────────────────────────────────────────────────────
tr = rollout_episode(
    ToyPricingEnv(pomdp=True),
    scripted_policy(['TOOL {"name":"lookup","key":"banana"}', "ANSWER 0.5"]),
)
assert len(tr) == 2 and tr[-1].done and abs(tr[-1].reward - 1.0) < 1e-9
print("rollout_episode ✓  scripted_policy ✓")


---
## 5 · Credit assignment: returns vs a simple baseline

**Monte Carlo advantages** (toy, per-step): subtract the mean of $\{G_t\}$ from each $G_t$.  
This is *not* a learned critic — it is a variance-reduction trick so early tool steps get non-zero learning signal when the episode succeeds.

Compare against **terminal-only** shaping intuition: if only the last reward is non-zero, all $G_t$ are identical (for $\gamma=1$), so centering still yields zero advantage — highlighting the need for **intermediate shaping** or **value baselines**.


In [ ]:
def centered_mc_advantages(rewards: List[float], gamma: float) -> List[float]:
    """A_t = G_t - mean(G) where G_t are Monte Carlo returns."""
    G = compute_mc_returns(rewards, gamma)
    if not G:
        return []
    mu = float(np.mean(G))
    return [g - mu for g in G]


def all_close_zero(xs: List[float], eps: float = 1e-9) -> bool:
    return all(abs(x) < eps for x in xs)


# ── Sanity checks ─────────────────────────────────────────────────────────
adv_sparse = centered_mc_advantages([0.0, 0.0, 1.0], gamma=1.0)
assert all_close_zero(adv_sparse), "Terminal-only + γ=1 → constant G_t → zero centered advantages"

adv_shaped = centered_mc_advantages([0.1, 0.1, 1.0], gamma=1.0)
assert not all_close_zero(adv_shaped)
assert abs(np.mean(adv_shaped)) < 1e-6
print("centered_mc_advantages ✓")


---
## 6 · Tiny policy gradient on bag-of-actions (optional bridge to Stage 1)

We wrap macro-actions `{LOOKUP, ANSWER}` in a logits table and take one REINFORCE step on a **batched** rollout of the toy MDP where the observation is ignored (tabular bandit-style). This shows how **trajectory return** plugs into `∇ log π` without involving Transformers.


In [ ]:
def macro_bandit_reinforce_step(
    logits:     np.ndarray,  # (A,) mutable in-place
    action_idx: int,
    advantage:  float,
    lr:         float = 0.1,
) -> float:
    """
    One REINFORCE step on softmax policy: ∇_θ log π(a) = e_a − π (for logits θ).
    Update: θ ← θ + lr * A * (e_a − π).
    Returns scalar loss = −A * log π(a) before the update.
    """
    theta = logits.astype(np.float64, copy=False)
    theta_max = theta.max()
    exps = np.exp(theta - theta_max)
    pi = exps / exps.sum()
    logp = np.log(pi[action_idx] + 1e-12)
    loss = float(-(advantage * logp))
    one_hot = np.zeros_like(pi)
    one_hot[action_idx] = 1.0
    # Gradient descent on L = -A log π(a):  θ -= lr * (-A (e_a - π)) = θ + lr A (e_a - π)
    theta += lr * advantage * (one_hot - pi)
    logits[:] = theta.astype(logits.dtype)
    return loss


# ── Sanity checks ─────────────────────────────────────────────────────────
np.random.seed(0)
lg = np.zeros(3, dtype=np.float64)
loss0 = macro_bandit_reinforce_step(lg, action_idx=1, advantage=1.0, lr=0.5)
assert isinstance(loss0, float)
ex = np.exp(lg - lg.max())
p = ex / ex.sum()
assert p[1] > 1.0 / 3.0, "Positive advantage on action 1 should increase its probability"
print("macro_bandit_reinforce_step ✓")
